# When Confidence Meets Cost — corrected reproducible analysis
Timestamp-aware renewable forecasting, empirical interval calibration and solar procurement.
Separate experiments examine SHAP attribution and simulated seasonal-client federation.
Train before 2020; tune on 2020; calibrate on 2021; evaluate January–August 2022.
Additional rolling-origin solar evaluations and repeated-seed federation are included.
Synthetic wind is an uncapped aerodynamic proxy. Neither exchangeability, conditional
shortfall guarantees nor privacy protection is established by these experiments.
All fitted models, tables, prediction arrays and generated plots are exported to artifacts/.
Dataset weather availability and astronomy provenance are assumptions requiring source documentation.

In [ ]:
import os, json, warnings, sys, platform, hashlib
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import lightgbm as lgb
from sklearn.linear_model import Ridge, LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from threadpoolctl import threadpool_limits
threadpool_limits(limits=4)
plt.rcParams.update({'figure.dpi':110,'font.size':10,'axes.grid':True,'grid.alpha':.3})

# --- Repo-relative paths (works whether run from repo root or notebooks/) ---
HERE = Path.cwd()
ROOT = HERE if (HERE / 'data').exists() else HERE.parent
sys.path.insert(0, str(ROOT / 'src'))          # make src/ modules importable

# Locate the dataset: data/Renewable.csv (preferred) or repo-root Renewable.csv
_candidates = [ROOT/'data'/'Renewable.csv', ROOT/'Renewable.csv', Path('Renewable.csv')]
RAW = next((str(p.resolve()) for p in _candidates if p.exists()), str((ROOT/'data'/'Renewable.csv').resolve()))

# --- Output layout: figures and tables in their own subfolders ---
OUT    = ROOT / 'output'
FIGS   = OUT / 'figures'
TABLES = OUT / 'tables'
for _p in (OUT, FIGS, TABLES):
    _p.mkdir(parents=True, exist_ok=True)

STEP_MIN, STEPS_PER_HOUR, STEPS_PER_DAY = 15, 4, 96

def savefig(name):
    plt.savefig(FIGS/name, dpi=180, bbox_inches='tight'); plt.close()

def table(name, d):
    d.to_csv(TABLES/(name+'.csv')); display(d)

print('Dataset :', RAW, '(exists)' if Path(RAW).exists() else '(MISSING - see data/README.md)')
print('Figures :', FIGS)
print('Tables  :', TABLES)


## 1. Data loading and physics-based synthetic wind

In [ ]:
def load():
    df = pd.read_csv(RAW)
    df["Time"] = pd.to_datetime(df["Time"])
    df = df.sort_values("Time").reset_index(drop=True)
    df.rename(columns={"Energy delta[Wh]": "solar"}, inplace=True)
    # physics-derived synthetic wind energy (Wh per 15-min): power-law wind profile,
    # ideal-gas air density, uncapped cubic aerodynamic proxy (not a rated turbine curve)
    alpha, R, r, Cp, eta = 0.14, 287.05, 40.0, 0.40, 0.90
    v_hub = df["wind_speed"] * (80.0/10.0)**alpha
    rho = (df["pressure"]*100.0)/(R*(df["temp"]+273.15))
    P = 0.5*rho*np.pi*r**2*v_hub**3*Cp*eta            # Watts
    df["wind"] = (P*0.25).clip(lower=0)               # Wh in a 15-min interval
    df["v_hub"], df["air_density"] = v_hub, rho
    return df

df = load()
print("rows:", len(df), "| span:", df.Time.min(), "->", df.Time.max())
print("solar zeros: %.1f%%" % (100*(df.solar==0).mean()))
df[["Time","solar","wind","GHI","temp","pressure","wind_speed","clouds_all"]].head()
expected=pd.date_range(df.Time.min(),df.Time.max(),freq='15min')
missing=expected.difference(pd.DatetimeIndex(df.Time))
data_audit={'rows':len(df),'start':str(df.Time.min()),'end':str(df.Time.max()),
            'missing_intervals':len(missing),'zero_fraction':float((df.solar==0).mean()),
            'daylight_zeros':int(((df.solar==0)&(df.isSun==1)).sum()),
            'max_wind_power_W':float(df.wind.max()/0.25),
            'sha256':hashlib.sha256(Path(RAW).read_bytes()).hexdigest()}
(OUT/'data_audit.json').write_text(json.dumps(data_audit,indent=2))

## 2. Causal feature engineering and blocked temporal split

In [ ]:
WEATHER=['GHI','temp','pressure','humidity','wind_speed','rain_1h','snow_1h','clouds_all']
ASTRO=['isSun','sunlightTime','dayLength','SunlightTime/daylength','hour','month']
def build_features(df,target,horizon,regime='endogenous'):
    """15-minute elapsed-time grid; reject incomplete history and origin-target paths.
    Current weather is assumed available at issue time; future astronomy is assumed known.
    No interpolation of targets or history across outages is performed.
    """
    d=df.set_index('Time').reindex(pd.date_range(df.Time.min(),df.Time.max(),freq='15min'))
    y=d[target].astype(float); feat=pd.DataFrame(index=d.index)
    for L in [0,1,2,3,4,96]:feat[f'{target}_lag{L}']=y.shift(L)
    feat[f'{target}_rmean_1h']=y.rolling(4,min_periods=4).mean()
    feat[f'{target}_rmean_1d']=y.rolling(96,min_periods=96).mean()
    feat[f'{target}_rstd_1h']=y.rolling(4,min_periods=4).std()
    for c in WEATHER:feat[c+'_now']=d[c]
    for c in ASTRO:feat[c+'_fut']=d[c].shift(-horizon)
    hr=d.hour.shift(-horizon); doy=pd.Series(d.index.dayofyear,index=d.index).shift(-horizon)
    feat['hour_sin']=np.sin(2*np.pi*hr/24);feat['hour_cos']=np.cos(2*np.pi*hr/24)
    feat['doy_sin']=np.sin(2*np.pi*doy/365.25);feat['doy_cos']=np.cos(2*np.pi*doy/365.25)
    if regime=='nwp':
        for c in WEATHER:feat[c+'_fut']=d[c].shift(-horizon)
    tgt=y.shift(-horizon)
    # Includes lag 96 and all history between it and the forecast origin.
    history=y.notna().rolling(97,min_periods=97).sum().eq(97)
    forward=y.notna().rolling(horizon+1,min_periods=horizon+1).sum().shift(-horizon).eq(horizon+1)
    mask=feat.notna().all(axis=1)&tgt.notna()&history&forward
    times=d.index+pd.Timedelta(minutes=15*horizon)
    return feat.loc[mask],tgt.loc[mask].to_numpy(),times[mask].to_numpy()
def chrono_split(times,test_year=2022):
    t=pd.DatetimeIndex(times)
    a=pd.Timestamp(f'{test_year-2}-01-01');b=pd.Timestamp(f'{test_year-1}-01-01');c=pd.Timestamp(f'{test_year}-01-01');e=pd.Timestamp(f'{test_year+1}-01-01')
    return np.asarray(t<a),np.asarray((t>=a)&(t<b)),np.asarray((t>=b)&(t<c)),np.asarray((t>=c)&(t<e))
def rmse(a,b):return float(np.sqrt(np.mean((np.asarray(a)-np.asarray(b))**2)))
def mae(a,b):return float(np.mean(np.abs(np.asarray(a)-np.asarray(b))))
def r2(a,b):
    a,b=np.asarray(a),np.asarray(b);return float(1-np.sum((a-b)**2)/np.sum((a-a.mean())**2))
def fitted(X,y,tr,tu,seed=42,quantile=None,n_estimators=600):
    m=lgb.LGBMRegressor(objective='regression' if quantile is None else 'quantile',
        **({} if quantile is None else {'alpha':quantile}),n_estimators=n_estimators,
        learning_rate=.05,num_leaves=63 if quantile is None else 47,subsample=.8,
        colsample_bytree=.8,min_child_samples=50,random_state=seed,n_jobs=4,verbosity=-1)
    m.fit(X.loc[tr].to_numpy(),y[tr],eval_set=[(X.loc[tu].to_numpy(),y[tu])],callbacks=[lgb.early_stopping(40,verbose=False)])
    return m
X,y,t=build_features(df,'solar',4);tr,tu,ca,te=chrono_split(t)
print('features:',len(X.columns),'train/tune/calibrate/test:',*[int(x.sum()) for x in [tr,tu,ca,te]])
(OUT/'split_counts.json').write_text(json.dumps({k:int(v.sum()) for k,v in zip(['train','tune','calibration','test'],[tr,tu,ca,te])},indent=2))

## 3. Stage 1 — the contemporaneous-estimation & leakage diagnostic
Reproduces the source setup (weather at *t* → energy at *t*) and shows how much of the
reported $R^2\approx0.91$ comes from (a) GHI alone and (b) random-split leakage.

In [ ]:
from sklearn.model_selection import train_test_split
d = load().dropna(subset=["solar","GHI"]).reset_index(drop=True)
Xc = d[WEATHER].values; yc = d["solar"].values

# (a) GHI-only linear model
from sklearn.linear_model import LinearRegression
ghi = d[["GHI"]].values
lin = LinearRegression().fit(ghi, yc)
print("GHI-only linear R2:                 %.3f" % r2(yc, lin.predict(ghi)))

# (b) full contemporaneous GBR, random vs chronological split
Xtr,Xte_,ytr,yte_ = train_test_split(Xc,yc,test_size=0.4,random_state=42)
g = lgb.LGBMRegressor(n_estimators=400,learning_rate=0.05,num_leaves=63,random_state=42,verbose=-1,n_jobs=4).fit(Xtr,ytr)
print("Contemporaneous GBR, RANDOM split:  %.3f" % r2(yte_, g.predict(Xte_)))
cut = int(0.6*len(d))
g2 = lgb.LGBMRegressor(n_estimators=400,learning_rate=0.05,num_leaves=63,random_state=42,verbose=-1,n_jobs=4).fit(Xc[:cut],yc[:cut])
print("Contemporaneous GBR, CHRONO split:  %.3f" % r2(yc[cut:], g2.predict(Xc[cut:])))

# duplicate / leakage rates
dup = d[WEATHER].duplicated().mean()
tr_idx,te_idx = train_test_split(np.arange(len(d)),test_size=0.4,random_state=42)
train_rows = set(map(tuple, np.round(Xc[tr_idx],6)))
twins = np.mean([tuple(r) in train_rows for r in np.round(Xc[te_idx],6)])
print("\nExact-duplicate feature rows:       %.1f%%" % (100*dup))
print("Test rows with a twin in train:     %.1f%%" % (100*twins))
diagnostic=pd.DataFrame({'R2':[r2(yc,lin.predict(ghi)),r2(yte_,g.predict(Xte_)),r2(yc[cut:],g2.predict(Xc[cut:]))]},index=['GHI in-sample','GBM random test','GBM chronological test'])
table('diagnostic',diagnostic)
fig,ax=plt.subplots(figsize=(6,3.8));ax.bar(range(3),diagnostic.R2,color=['#95a5a6','#c0392b','#1f77b4']);ax.set_ylim(0,1)
ax.set_xticks(range(3),['GHI linear\nin-sample','GBM random\ntest','GBM temporal\ntest']);ax.set_ylabel('$R^2$');ax.set_title('Contemporaneous estimation diagnostic')
for i,v in enumerate(diagnostic.R2):ax.text(i,v+.01,f'{v:.3f}',ha='center')
plt.tight_layout();savefig('image16.png')
(OUT/'duplicate_rates.json').write_text(json.dumps({'duplicate_features':float(dup),'random_test_overlap':float(twins)},indent=2))

Repeated weather vectors and split-period differences make random and temporal
scores noncomparable as a causal leakage estimate. GHI is an in-sample diagnostic.
Subsequent forecasting uses elapsed-time horizons and rejects gap-crossing windows.

## 4. Stage 2 — model benchmark + feature-group ablation (solar, 1 h)

In [ ]:
X,y,t=build_features(df,'solar',4);tr,tu,ca,te=chrono_split(t)
Xtr,ytr=X.loc[tr].to_numpy(),y[tr];Xte,yte=X.loc[te].to_numpy(),y[te];cols=list(X.columns)
bench=[]; PRED={}
persist=X.loc[te,'solar_lag0'].to_numpy()
seasonal=df.set_index('Time').solar.reindex(pd.DatetimeIndex(t[te])-pd.Timedelta(days=1)).to_numpy()
assert np.isfinite(seasonal).all()
for name,p in [('Persistence',persist),('Seasonal-naive (24h)',seasonal)]:
    PRED[name]=p;bench.append((name,rmse(yte,p),mae(yte,p),r2(yte,p)))
sc=StandardScaler().fit(Xtr)
ridge=Ridge(alpha=1).fit(sc.transform(Xtr),ytr)
rf=RandomForestRegressor(n_estimators=100,max_depth=18,n_jobs=4,random_state=42).fit(Xtr,ytr)
gb=fitted(X,y,tr,tu)
for name,m,xx in [('Ridge (linear)',ridge,sc.transform(Xte)),('Random Forest',rf,Xte),('LightGBM',gb,Xte)]:
    p=np.clip(m.predict(xx),0,None);PRED[name]=p;bench.append((name,rmse(yte,p),mae(yte,p),r2(yte,p)))
gb.booster_.save_model(str(OUT/'point_solar_1h.txt'))
bench_df=pd.DataFrame(bench,columns=['model','RMSE','MAE','R2']).set_index('model');table('benchmark',bench_df)
AR=[c for c in cols if c.startswith('solar_')];WN=[c for c in cols if c.endswith('_now')];AF=[c for c in cols if c.endswith('_fut') or c.endswith('_sin') or c.endswith('_cos')]
sets={'AR only':AR,'AR+weather(now)':AR+WN,'AR+astro(future)':AR+AF,'Full':cols};abl=[]
for name,cs in sets.items():
    mm=fitted(X[cs],y,tr,tu,n_estimators=500);pp=np.clip(mm.predict(X.loc[te,cs].to_numpy()),0,None)
    abl.append((name,len(cs),rmse(yte,pp),r2(yte,pp)));mm.booster_.save_model(str(OUT/('ablation_'+name.replace('+','_').replace('/','_')+'.txt')))
abl_df=pd.DataFrame(abl,columns=['feature set','#feat','RMSE','R2']).set_index('feature set');table('ablation',abl_df)
pd.DataFrame({'time':t[te],'y':yte,**PRED}).to_csv(OUT/'point_predictions.csv',index=False)
# Paired moving-day bootstrap; blocks respect missing calendar days.
day_errors=pd.DataFrame({'date':pd.DatetimeIndex(t[te]).normalize(),'GB':(yte-PRED['LightGBM'])**2,'RF':(yte-PRED['Random Forest'])**2}).groupby('date').agg(['sum','count'])
daily=day_errors.reindex(pd.date_range(day_errors.index.min(),day_errors.index.max(),freq='D')).fillna(0)
boot=np.random.default_rng(2026);diff=[];L=7;N=len(daily)
for _ in range(1000):
    starts=boot.integers(0,N,size=int(np.ceil(N/L)));idx=np.concatenate([(s+np.arange(L))%N for s in starts])[:N];b=daily.iloc[idx]
    diff.append(np.sqrt(b[('GB','sum')].sum()/b[('GB','count')].sum())-np.sqrt(b[('RF','sum')].sum()/b[('RF','count')].sum()))
ci=np.quantile(diff,[.025,.975]);(OUT/'paired_bootstrap.json').write_text(json.dumps({'GB_minus_RF_RMSE':rmse(yte,PRED['LightGBM'])-rmse(yte,PRED['Random Forest']),'CI95':ci.tolist(),'block_days':7,'replicates':1000},indent=2))
import joblib
joblib.dump({"ridge":ridge,"random_forest":rf,"scaler":sc,"columns":cols},OUT/"point_baselines.joblib")


In [ ]:
fig,ax=plt.subplots(1,2,figsize=(11,3.8))
ax[0].barh(bench_df.index, bench_df["R2"],
           color=["#95a5a6","#95a5a6","#7f8c8d","#f39c12","#1f77b4"])
ax[0].set_xlim(0,1); ax[0].invert_yaxis(); ax[0].set_xlabel("Test $R^2$")
ax[0].set_title("Model benchmark (solar, 1 h)")
for i,v in enumerate(bench_df["R2"]): ax[0].text(v+0.004,i,f"{v:.3f}",va="center",fontsize=8)
ax[1].bar(range(len(abl_df)), abl_df["R2"], color="#8e44ad")
ax[1].set_xticks(range(len(abl_df))); ax[1].set_xticklabels(abl_df.index,rotation=20,ha="right")
ax[1].set_ylim(0,1); ax[1].set_ylabel("Test $R^2$"); ax[1].set_title("Feature-group ablation")
for i,v in enumerate(abl_df["R2"]): ax[1].text(i,v+0.001,f"{v:.3f}",ha="center",fontsize=8)
plt.tight_layout(); savefig('image17.png')

## 5. Stage 3 — genuine multi-horizon forecasting
Endogenous (no future weather) vs NWP-oracle (perfect future weather), for real solar and
synthetic wind at 15 min / 1 h / 3 h / 6 h.

In [ ]:
HORIZONS={'15min':1,'1h':4,'3h':12,'6h':24}
sweep=[]
for target in ['solar','wind']:
    for regime in ['endogenous','nwp']:
        for hn,h in HORIZONS.items():
            X,y,t=build_features(df,target,h,regime);tr,tu,ca,te=chrono_split(t);m=fitted(X,y,tr,tu)
            pred=np.clip(m.predict(X.loc[te].to_numpy()),0,None);persist=X.loc[te,f'{target}_lag0'].to_numpy()
            sweep.append(dict(target=target,regime=regime,horizon=hn,R2=r2(y[te],pred),RMSE=rmse(y[te],pred),skill=1-rmse(y[te],pred)/rmse(y[te],persist)))
            m.booster_.save_model(str(OUT/f'point_{target}_{regime}_{hn}.txt'))
            pd.DataFrame({'time':t[te],'y':y[te],'prediction':pred,'persistence':persist}).to_csv(OUT/f'point_{target}_{regime}_{hn}.csv',index=False)
sweep_df=pd.DataFrame(sweep);table('horizons',sweep_df)
fig,axs=plt.subplots(1,2,figsize=(11,3.8))
for ax,target in zip(axs,['solar','wind']):
    for regime,color in [('endogenous','#1f77b4'),('nwp','#f39c12')]:
        z=sweep_df[(sweep_df.target==target)&(sweep_df.regime==regime)]
        ax.plot([.25,1,3,6],z.R2,'-o',label=regime,color=color)
    ax.set_title(target+' forecasting');ax.set_xlabel('Horizon (hours)');ax.set_ylabel('Test $R^2$');ax.legend(fontsize=8)
plt.tight_layout();savefig('image18.png')

## 6. Stage 4 — Conformalized Quantile Regression (CQR) + probabilistic metrics

In [ ]:
QGRID=[.05,.1,.25,.5,.75,.9,.95];QLO,QMID,QHI,ALPHA=.05,.5,.95,.10
def pinball(y,p,q):
    d=y-p;return float(np.mean(np.maximum(q*d,(q-1)*d)))
def score_quantile(s,miscoverage):
    k=int(np.ceil((len(s)+1)*(1-miscoverage)))
    return float(np.sort(s)[k-1]) if k<=len(s) else np.inf
def winkler(y,lo,hi,a):return float(np.mean(hi-lo+2/a*np.maximum(lo-y,0)+2/a*np.maximum(y-hi,0)))
ARRS={};GRIDS={};prob_rows=[];rolling=[]
def run_prob(target,regime,horizon=4,test_year=2022,seed=42):
    X,y,t=build_features(df,target,horizon,regime);tr,tu,ca,te=chrono_split(t,test_year)
    ca_pred=[];te_pred=[]
    for q in QGRID:
        m=fitted(X,y,tr,tu,seed,q,400)
        ca_pred.append(np.clip(m.predict(X.loc[ca].to_numpy()),0,None));te_pred.append(np.clip(m.predict(X.loc[te].to_numpy()),0,None))
        m.booster_.save_model(str(OUT/f'quantile_{target}_{regime}_{test_year}_{q}.txt'))
    pc=np.column_stack(ca_pred);pt=np.column_stack(te_pred)
    crossing=float(np.any(np.diff(pt,axis=1)<0,axis=1).mean())
    # Rearrangement is fixed before calibration, then applied identically to test predictions.
    pc=np.sort(pc,axis=1);pt=np.sort(pt,axis=1)
    Q=score_quantile(np.maximum(pc[:,0]-y[ca],y[ca]-pc[:,-1]),ALPHA)
    lo=np.clip(pt[:,0]-Q,0,None);hi=np.maximum(pt[:,-1]+Q,lo)
    lower_Q=score_quantile(pc[:,0]-y[ca],.05);lower=np.clip(pt[:,0]-lower_Q,0,None)
    day=X.loc[te,'isSun_fut'].to_numpy()==1;yc=y[te]
    cov=lambda a,b,mask:float(np.mean((yc[mask]>=a[mask])&(yc[mask]<=b[mask])))
    rec={'target':target,'regime':regime,'test_year':test_year,'seed':seed,
         'picp_raw':cov(pt[:,0],pt[:,-1],np.ones(len(yc),bool)),'picp_cqr':cov(lo,hi,np.ones(len(yc),bool)),
         'picp_day_raw':cov(pt[:,0],pt[:,-1],day),'picp_day_cqr':cov(lo,hi,day),
         'pinaw_cqr':float(np.mean(hi-lo)/(yc.max()-yc.min())),'width_Wh':float(np.mean(hi-lo)),
         'width_over_positive_mean':float(np.mean(hi-lo)/np.mean(yc[yc>0])),
         'winkler_cqr':winkler(yc,lo,hi,ALPHA),'raw_crossing_rate':crossing,
         'lower_raw_failure':float(np.mean(yc<pt[:,0])),'lower_one_sided_failure':float(np.mean(yc<lower)),
         'lower_day_failure':float(np.mean(yc[day]<lower[day]))}
    # Integrate pinball over [.05,.95]; tails explicitly omitted, not full CRPS.
    losses=np.array([pinball(yc,pt[:,j],q) for j,q in enumerate(QGRID)])
    rec['truncated_quantile_integral']=float(2*np.trapezoid(losses,QGRID))
    arr={'time':pd.DatetimeIndex(t[te]),'y':yc,'med':pt[:,3],'lo':lo,'hi':hi,'lower':lower,'raw_lo':pt[:,0],'pt':pt,'day':day}
    pred=pd.DataFrame({'time':t[te],'y':yc,'day':day,'median':arr['med'],'raw_lo':arr['raw_lo'],'cqr_lo':lo,'cqr_hi':hi,'one_sided_lo':lower})
    for j,q in enumerate(QGRID):pred[f'q{q}']=pt[:,j]
    pred.to_csv(OUT/f'prob_predictions_{target}_{regime}_{test_year}_{seed}.csv',index=False)
    seasonal=pred.assign(season=(pd.DatetimeIndex(t[te]).month%12//3).astype(int)).groupby('season').apply(lambda d:pd.Series({'n':len(d),'coverage':((d.y>=d.cqr_lo)&(d.y<=d.cqr_hi)).mean(),'lower_failure':(d.y<d.one_sided_lo).mean(),'day_lower_failure':(d.loc[d.day,'y']<d.loc[d.day,'one_sided_lo']).mean()}),include_groups=False)
    seasonal.to_csv(OUT/f'seasonal_coverage_{target}_{regime}_{test_year}.csv')
    if test_year==2022 and seed==42:
        ARRS[(target,regime)]=arr;GRIDS[(target,regime)]={'pte':{q:pt[:,j] for j,q in enumerate(QGRID)},'yte':yc,'day':day}
    return rec
for target in ['solar','wind']:
    for regime in ['endogenous','nwp']:prob_rows.append(run_prob(target,regime))
prob_df=pd.DataFrame(prob_rows);table('probabilistic',prob_df)
# Rolling-origin intervals use their own disjoint training/tuning/calibration slices.
for year in [2020,2021]:rolling.append(run_prob('solar','endogenous',test_year=year))
rolling.append(prob_rows[0]);table('rolling_coverage',pd.DataFrame(rolling))
roll_point=[]
for year in [2020,2021,2022]:
    X,y,t=build_features(df,'solar',4);tr,tu,ca,te=chrono_split(t,year)
    for seed in [42,43,44]:
        m=fitted(X,y,tr,tu,seed);m.booster_.save_model(str(OUT/f'rolling_solar_{year}_{seed}.txt'));p=np.clip(m.predict(X.loc[te].to_numpy()),0,None)
        pd.DataFrame({'time':t[te],'y':y[te],'prediction':p}).to_csv(OUT/f'rolling_solar_{year}_{seed}.csv',index=False)
        roll_point.append({'test_year':year,'seed':seed,'R2':r2(y[te],p),'RMSE':rmse(y[te],p),'skill':1-rmse(y[te],p)/rmse(y[te],X.loc[te,'solar_lag0'].to_numpy())})
table('rolling_point',pd.DataFrame(roll_point))

In [ ]:
# Probabilistic forecast example: 4-day window (solar, endogenous, 1 h)
a=ARRS[("solar","endogenous")]; tt=a["time"]
mask=(tt>=pd.Timestamp("2022-06-10"))&(tt<pd.Timestamp("2022-06-14"))
plt.figure(figsize=(11,3.6))
plt.fill_between(tt[mask],a["lo"][mask],a["hi"][mask],alpha=.3,color="#1f77b4",label="Nominal 90% CQR interval")
plt.plot(tt[mask],a["med"][mask],color="#1f77b4",lw=1.4,label="median forecast")
plt.plot(tt[mask],a["y"][mask],color="black",lw=1,label="actual")
plt.title("1-hour-ahead probabilistic solar forecast (January–August 2022 test)")
plt.ylabel("Energy (Wh)"); plt.legend(fontsize=8); plt.tight_layout(); savefig('image19.png')

## 7. Reliability diagram (quantile calibration)

In [ ]:
g=GRIDS[("solar","endogenous")]; yte=g["yte"]; day=g["day"]
emp_full=[float(np.mean(yte<=g["pte"][q])) for q in QGRID]
emp_day =[float(np.mean(yte[day]<=g["pte"][q][day])) for q in QGRID]
plt.figure(figsize=(5,4.4))
plt.plot([0,1],[0,1],"k--",lw=1,label="Ideal")
plt.plot(QGRID,emp_full,"-o",color="#c0392b",label="All hours (zero-inflated)")
plt.plot(QGRID,emp_day, "-s",color="#1f77b4",label="Daytime only")
plt.xlabel("Nominal quantile level"); plt.ylabel("Empirical coverage")
plt.title("Quantile reliability (solar, 1 h)"); plt.legend(fontsize=8); plt.tight_layout(); savefig('image21.png')
print("daytime coverage:", {q:round(c,3) for q,c in zip(QGRID,emp_day)})
table('quantile_reliability',pd.DataFrame({'q':QGRID,'overall':emp_full,'daylight':emp_day}))
fig,ax=plt.subplots(1,2,figsize=(11,3.8));x=np.arange(len(prob_df));w=.35
ax[0].bar(x-w/2,prob_df.picp_raw,w,label='raw');ax[0].bar(x+w/2,prob_df.picp_cqr,w,label='CQR');ax[0].axhline(.9,color='black',ls='--');ax[0].set_ylim(0,1);ax[0].legend(fontsize=8);ax[0].set_title('Empirical interval coverage')
ax[1].bar(x,prob_df.pinaw_cqr,color='#8e44ad');ax[1].set_ylim(bottom=0);ax[1].set_title('Normalized interval width')
for a in ax:a.set_xticks(x,['Solar\nEndo','Solar\nOracle','Wind\nEndo','Wind\nOracle'])
plt.tight_layout();savefig('image20.png')

## 8. SHAP attribution
Explain endogenous and perfect-weather-oracle point models. Attribution describes the
fitted prediction function; it is not a causal validation or explanation of procurement decisions.

In [ ]:
import shap
PRETTY={"GHI_fut":"GHI (t+h)","GHI_now":"GHI (t)","wind_speed_fut":"wind speed (t+h)",
 "wind_speed_now":"wind speed (t)","temp_fut":"temp (t+h)","clouds_all_fut":"clouds (t+h)",
 "isSun_fut":"isSun (t+h)","SunlightTime/daylength_fut":"sunlight frac (t+h)","solar_lag0":"solar (t)",
 "solar_rmean_1h":"solar 1h-mean","pressure_fut":"pressure (t+h)","wind_lag0":"wind (t)",
 "dayLength_fut":"dayLength (t+h)","humidity_fut":"humidity (t+h)"}
def shap_explain(target,regime="nwp"):
    X,y,t=build_features(df,target,4,regime); tr,tu,ca,te=chrono_split(t)
    m=lgb.LGBMRegressor(objective="regression",n_estimators=500,learning_rate=0.05,num_leaves=63,
        subsample=0.8,colsample_bytree=0.8,min_child_samples=50,random_state=42,verbose=-1,n_jobs=4)
    m.fit(X[tr].values,y[tr],eval_set=[(X[tu].values,y[tu])],callbacks=[lgb.early_stopping(40,verbose=False)])
    idx=np.where(te)[0]; samp=np.random.default_rng(0).choice(idx,size=min(3000,len(idx)),replace=False)
    Xs=X.iloc[samp]; sv=shap.TreeExplainer(m).shap_values(Xs)
    shap.summary_plot(sv,Xs,feature_names=[PRETTY.get(c,c) for c in X.columns],
                      max_display=10,show=False,plot_size=(6,4))
    plt.title(f"SHAP — {'Solar' if target=='solar' else 'Wind'} model",fontsize=10)
    plt.tight_layout(); savefig(('image22.png' if target=='solar' else 'image23.png') if regime=='nwp' else f'shap_{target}_{regime}.png')
    pd.DataFrame({'feature':X.columns,'mean_abs_shap':np.abs(sv).mean(0)}).sort_values('mean_abs_shap',ascending=False).to_csv(OUT/f'shap_{target}_{regime}.csv',index=False)
shap_explain("solar")
shap_explain("wind")
shap_explain("solar","endogenous")
shap_explain("wind","endogenous")

## 9. Simulated federated point forecasting
Five seasonal partitions of one site, not five measured plants. Three seeds use identical
initial parameters and matched 40-epoch observation budgets for central/local/FedAvg.
Global scaling moments are computed by aggregation of client sums and squared sums
on the same 48,000-row subsample. This simulation implements neither secure aggregation
nor differential privacy; the shared moments and model weights are not privacy protected.
It is independent of the quantile and procurement models.

In [ ]:
X,y,t=build_features(df,'solar',4);tr,tu,ca,te=chrono_split(t)
Xtr,ytr=X.loc[tr].to_numpy().astype(np.float32),y[tr].astype(np.float32)
Xte,yte=X.loc[te].to_numpy().astype(np.float32),y[te].astype(np.float32)
months=X.loc[tr,'month_fut'].to_numpy().astype(int);d_in=Xtr.shape[1];Hd=64;K=5
groups={0:[12,1,2],1:[3,4],2:[5,6],3:[7,8],4:[9,10,11]}
def fwd(P,A):
    h=np.maximum(A@P['W1']+P['b1'],0);return (h@P['W2']+P['b2']).ravel(),h
def sgd(P,A,b,rng,epochs=1,bs=512,lr=.02):
    for _ in range(epochs):
        idx=rng.permutation(len(A))
        for s in range(0,len(A),bs):
            ii=idx[s:s+bs];xb,yb=A[ii],b[ii];o,h=fwd(P,xb);g=(o-yb)/len(ii)
            gW2=h.T@g[:,None];gb2=g.sum(keepdims=True);gh=(g[:,None]@P['W2'].T)*(h>0)
            for k,v in [('W1',xb.T@gh),('b1',gh.sum(0)),('W2',gW2),('b2',gb2.ravel())]:P[k]-=.02*v
    return P
copy=lambda P:{k:v.copy() for k,v in P.items()}
federation_rows=[];client_rows=[]
for seed in [42,43,44]:
    rng=np.random.default_rng(seed);sub=rng.choice(len(Xtr),size=min(48000,len(Xtr)),replace=False)
    Xs,ys,ms=Xtr[sub],ytr[sub],months[sub];parts_ni=[np.where(np.isin(ms,g))[0] for g in groups.values()];parts_iid=np.array_split(rng.permutation(len(Xs)),5)
    # Client-aggregated moments, limited to exactly the training subsample.
    sums=[Xs[p].astype(float).sum(0) for p in parts_ni];squares=[(Xs[p].astype(float)**2).sum(0) for p in parts_ni]
    mu=sum(sums)/len(Xs);sd=np.sqrt(np.maximum(sum(squares)/len(Xs)-mu**2,0))+1e-6
    ymu=sum(float(ys[p].sum()) for p in parts_ni)/len(ys);ysd=np.sqrt(sum(float((ys[p].astype(float)**2).sum()) for p in parts_ni)/len(ys)-ymu**2)
    Zs=((Xs-mu)/sd).astype(np.float32);Zte=((Xte-mu)/sd).astype(np.float32);yz=(ys-ymu)/ysd
    initial={'W1':rng.normal(0,np.sqrt(2/d_in),(d_in,Hd)).astype(np.float32),'b1':np.zeros(Hd,np.float32),'W2':rng.normal(0,np.sqrt(2/Hd),(Hd,1)).astype(np.float32),'b2':np.zeros(1,np.float32)}
    def ev(P,mask=None):
        pred=np.clip(fwd(P,Zte)[0]*ysd+ymu,0,None)
        return r2(yte if mask is None else yte[mask],pred if mask is None else pred[mask])
    def fedavg(parts):
        G=copy(initial);w=np.array([len(p) for p in parts])/len(Xs);curve=[];rounds=[]
        local_rng=[np.random.default_rng(seed*100+k) for k in range(5)]
        for r in range(40):
            localP=[sgd(copy(G),Zs[p],yz[p],local_rng[k]) for k,p in enumerate(parts)]
            G={k:sum(w[j]*localP[j][k] for j in range(5)) for k in G}
            if r%2==0 or r==39:curve.append(ev(G));rounds.append(r+1)
        return G,curve,rounds
    centralP=sgd(copy(initial),Zs,yz,np.random.default_rng(seed*1000),epochs=40);central=ev(centralP)
    localP=[sgd(copy(initial),Zs[p],yz[p],np.random.default_rng(seed*100+k),epochs=40) for k,p in enumerate(parts_ni)];local=[ev(P) for P in localP]
    G_ni,curve_ni,rounds_ni=fedavg(parts_ni);G_iid,curve_iid,rounds_iid=fedavg(parts_iid)
    vals={'Local-only (worst)':min(local),'Local-only (mean)':float(np.mean(local)),'Local-only (best)':max(local),'FedAvg non-IID':ev(G_ni),'FedAvg IID':ev(G_iid),'Centralized':central}
    for name,value in vals.items():federation_rows.append({'seed':seed,'scheme':name,'R2':value})
    for k,g in groups.items():
        mask=np.isin(pd.DatetimeIndex(t[te]).month,g)
        if mask.sum()>1:
            for name,P in [('local',localP[k]),('FedAvg',G_ni),('central',centralP)]:client_rows.append({'seed':seed,'client':k,'scheme':name,'n':int(mask.sum()),'R2':ev(P,mask)})
    np.savez(OUT/f'federated_model_{seed}.npz',**G_ni,mu=mu,sd=sd,ymu=ymu,ysd=ysd)
    np.savez(OUT/f'central_model_{seed}.npz',**centralP,mu=mu,sd=sd,ymu=ymu,ysd=ysd)
    np.savez(OUT/f'federated_iid_model_{seed}.npz',**G_iid,mu=mu,sd=sd,ymu=ymu,ysd=ysd)
    pd.DataFrame({'round':rounds_ni,'nonIID':curve_ni,'IID':curve_iid}).to_csv(OUT/f'federation_curve_{seed}.csv',index=False)
    for k,P in enumerate(localP):np.savez(OUT/f'local_model_{seed}_{k}.npz',**P,mu=mu,sd=sd,ymu=ymu,ysd=ysd)
    pd.DataFrame({'time':t[te],'y':yte,'federated':np.clip(fwd(G_ni,Zte)[0]*ysd+ymu,0,None),'central':np.clip(fwd(centralP,Zte)[0]*ysd+ymu,0,None)}).to_csv(OUT/f'federation_predictions_{seed}.csv',index=False)
    if seed==42:first_curves=(curve_ni,curve_iid,rounds_ni,rounds_iid,central,local,ev(G_ni))
federation_df=pd.DataFrame(federation_rows);table('federation_seeds',federation_df)
fl=federation_df.groupby('scheme').R2.agg(['mean','std']);table('federation',fl)
table('federation_clients',pd.DataFrame(client_rows))
curve_ni,curve_iid,rounds_ni,rounds_iid,central,local,fed_score=first_curves

In [ ]:
fig,ax=plt.subplots(1,2,figsize=(11,3.8))
ax[0].plot(rounds_ni,curve_ni,"-o",color="#1f77b4",label="FedAvg (non-IID)")
ax[0].plot(rounds_iid,curve_iid,"-^",color="#16a085",label="FedAvg (IID)")
ax[0].axhline(central,ls="--",color="#2ca02c",label="Centralized"); ax[0].axhline(np.mean(local),ls=":",color="#c0392b",label="Local-only (mean)")
ax[0].set_xlabel("Communication round"); ax[0].set_ylabel("Test $R^2$"); ax[0].set_ylim(0,1)
ax[0].set_title("Federated training curve"); ax[0].legend(fontsize=8,loc="lower right")
vals=[min(local),np.mean(local),max(local),fed_score,central]
ax[1].bar(range(5),vals,color=["#e74c3c","#c0392b","#e67e22","#1f77b4","#2ca02c"])
ax[1].set_xticks(range(5)); ax[1].set_xticklabels(["Local\n(worst)","Local\n(mean)","Local\n(best)","FedAvg\n(non-IID)","Central"],fontsize=8)
ax[1].set_ylim(0,1); ax[1].set_ylabel("Test $R^2$"); ax[1].set_title("Simulated point-forecast comparison")
for i,v in enumerate(vals): ax[1].text(i,v+0.004,f"{v:.3f}",ha="center",fontsize=8)
plt.tight_layout(); savefig('image24.png')

## 10. Stylized solar procurement
The classical economic fractile depends on tariffs. Interpolate the monotone raw
quantile grid at that level; do not interpret two-sided CQR endpoints as a calibrated CDF.
Compare the median, raw 5% quantile, CQR lower endpoint, separately calibrated one-sided
lower bound, tariff-selected policy and optional 5% risk-cap surrogate. The latter is
empirical: no conditional chance-constraint guarantee follows from marginal calibration.
Policies are fixed before test outcomes. Perfect information is a descriptive reference.

In [ ]:
a=ARRS[('solar','endogenous')];tt=a['time'];yv=a['y'];med=a['med'];lo=a['lo'];day=a['day']
hr=(tt.hour+tt.minute/60).to_numpy().astype(float);shape=np.clip(np.sin((hr-6)/12*np.pi),0,None)
D=np.clip(800+2600*shape+np.random.default_rng(0).normal(0,80,len(tt)),300,None);p_s,p_g=.02,.12
def sim(commit,p_b):
    S=np.minimum(D,np.clip(commit,0,None));Gs=D-S;served=np.minimum(S,yv);deficit=np.maximum(S-yv,0)
    return {'cost':float(((served*p_s+Gs*p_g+deficit*p_b)/1000).sum()),'bal':float(deficit.sum()/1000),'short':float((deficit>1e-6).mean()),'short_day':float((deficit[day]>1e-6).mean())}
def quantile_at(beta):
    assert QGRID[0]<=beta<=QGRID[-1]
    j=int(np.searchsorted(QGRID,beta));j=min(max(j,1),len(QGRID)-1)
    w=(beta-QGRID[j-1])/(QGRID[j]-QGRID[j-1]);return (1-w)*a['pt'][:,j-1]+w*a['pt'][:,j]
rows=[];policy_arrays={'time':tt,'y':yv,'demand':D}
for mult in [1,2,3,5,8,10,15]:
    pb=p_g*mult;beta=np.clip((p_g-p_s)/(pb-p_s),0,1)
    tariff=D if beta==1 else quantile_at(beta)
    policies={'Median':med,'Raw05':a['raw_lo'],'CQRlower':lo,'OneSided05':a['lower'],'Tariff':tariff,'RiskCap05':np.minimum(tariff,a['lower']),'PerfectInfo':yv}
    rec={'m':mult,'p_bal':pb,'beta_star':beta}
    for name,commit in policies.items():
        metrics=sim(commit,pb)
        rec.update({name+'_'+k:v for k,v in metrics.items()});policy_arrays[f'{name}_{mult}']=np.minimum(D,np.clip(commit,0,None))
    rec['save_vs_median']=100*(rec['Median_cost']-rec['Tariff_cost'])/rec['Median_cost'];rows.append(rec)
disp_df=pd.DataFrame(rows).set_index('m');table('dispatch',disp_df)
pd.DataFrame(policy_arrays).to_csv(OUT/'dispatch_predictions.csv',index=False)
(OUT/'demand_audit.json').write_text(json.dumps({'demand_MWh':float(D.sum()/1e6),'n':len(D),'start':str(tt.min()),'end':str(tt.max())},indent=2))

In [ ]:
fig,ax=plt.subplots(1,2,figsize=(11,3.8))
for key,label,color in [('Median','Median','#c0392b'),('Tariff','Tariff-selected','#1f77b4'),('PerfectInfo','Perfect information','#2ca02c')]:ax[0].plot(disp_df.index,disp_df[key+'_cost'],'-o',label=label,color=color)
ax[0].set_xlabel('Imbalance multiplier');ax[0].set_ylabel('Realized cost ($)');ax[0].set_title('Stylized procurement sensitivity');ax[0].legend(fontsize=8)
keys=['Median','CQRlower','Tariff','PerfectInfo'];values=[disp_df.loc[10,k+'_cost'] for k in keys]
ax[1].barh(range(4),values,color=['#c0392b','#95a5a6','#1f77b4','#2ca02c']);ax[1].set_yticks(range(4),['Median','CQR lower','Tariff-selected','Perfect information']);ax[1].invert_yaxis();ax[1].set_xlabel('Cost ($)');ax[1].set_title('10× imbalance: fixed policies')
for i,v in enumerate(values):ax[1].text(v+15,i,f'{v:,.0f}',va='center',fontsize=8)
plt.tight_layout();savefig('image25.png')

## 11. Daylight calibration and causal online updates
Pooled calibration is retained as a baseline. Compare daylight/night strata, fixed
daylight calibration, 30/90-calendar-day rolling scores and adaptive 90-day scores
with feedback rates 0.005/0.020. Selection uses 2020 and 2021 only, requiring at least
89% daylight interval coverage and at most 5.5% daylight lower-bound failure in both
windows, then minimizing mean daylight Winkler score. These tolerances are validation
criteria, not guarantees. A fallback minimizes the worst criterion deficit.
At each one-hour issue time, use only scores whose target observations have arrived;
the final calibration-hour outcomes are unavailable to the first test forecast.
Adaptive interval/lower miscoverage states target 0.10/0.05 separately, update only
on revealed daylight forecast errors, and are clipped to [0.001, 0.5].
The 2022 window was inspected before this method was proposed, so its assessment is
exploratory. Group calibration does not establish exchangeability within a time series;
feedback-based average error control is not a per-decision conditional guarantee.
The tiny validation score difference between the selected adaptive method and rolling
calibration does not establish statistical superiority. Fresh data are still required.


In [ ]:
# Section 11 needs two user-supplied modules in src/:
#   src/daylight_calibration.py  (exposes run)
#   src/daylight_reporting.py    (exposes report)
# If they are absent the notebook still completes; Sections 1-10 are unaffected.
try:
    from daylight_calibration import run as compare_daylight
    from daylight_reporting import report as report_daylight
    daylight_summary, selected_calibration, daylight_predictions = compare_daylight(df, build_features, chrono_split)
    display(daylight_summary)
    daylight_dispatch, daylight_uncertainty = report_daylight(daylight_summary, selected_calibration, daylight_predictions)
except ModuleNotFoundError as e:
    print(f"[skipped] Section 11 requires src/daylight_calibration.py and src/daylight_reporting.py ({e}).")
    print("Add those modules to run the daylight-calibration experiment; the rest of the notebook is complete.")


## Summary and remaining limits
The daylight-specific adaptive configuration was selected using the earlier windows.
On the already-inspected 2022 window, daylight interval coverage is 90.0% versus pooled
87.3%; mean daylight width increases 3.1%. Daylight lower-bound failure is 4.98% versus
7.61%. At 10× imbalance, the daylight risk-cap policy costs about $3,788 versus $3,780
for the pooled risk cap, reducing daytime procurement shortfall from 7.59% to 4.90%.
These are exploratory observations, not fresh-test, conditional-risk or privacy guarantees.
Raw-quantile point reliability, point forecasting, wind results and federation retain
their separate interpretations. Source provenance/availability/licence remain unresolved.
